In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random
import os
import json

rng = np.random.default_rng(42)
random.seed(42)

N = 1500  # >= 1000 rows

states = ["TX","CA","FL","NY","IL","MA","GA","NC","WA","AZ"]
plan_types = ["HMO","PPO","EPO","POS"]
provider_specialties = ["FamilyMed","InternalMed","Ortho","Cardiology","Derm","Neuro","Oncology","ER","Radiology"]
place_of_service = ["Office","Inpatient","Outpatient","ER","Telehealth"]
diagnosis_groups = ["MSK","Cardio","Neuro","Onc","Derm","Resp","GI","Endo"]
procedure_groups = ["Imaging","Surgery","Labs","Therapy","Consult","DME","Rx"]

def pick_code(prefix, k=4):
    return prefix + "".join([str(random.randint(0,9)) for _ in range(k)])

def make_note(dx_grp, proc_grp, severity, suspicious):
    templates = [
        "Patient reports worsening symptoms related to {dx}. Ordered {proc}. Severity {sev}.",
        "Clinical note: {dx} condition. Requesting {proc}. Prior treatments tried. Severity {sev}.",
        "Member complaint: delays in approval. {dx} suspected. {proc} requested. Severity {sev}.",
        "Provider note: rule-out for {dx}. Needs {proc}. Severity {sev}.",
    ]
    t = random.choice(templates).format(dx=dx_grp, proc=proc_grp, sev=severity)
    if suspicious:
        t += " Billing pattern unusual; request urgent; documentation incomplete."
    return t

# --- Core columns ---
claim_id = [f"CLM{100000+i}" for i in range(N)]
member_id = [f"MBR{rng.integers(10000,99999)}" for _ in range(N)]
provider_id = [f"NPI{rng.integers(1000000000,9999999999)}" for _ in range(N)]

start_date = datetime(2025, 1, 1)
service_date = [start_date + timedelta(days=int(x)) for x in rng.integers(0, 365, size=N)]
received_date = [d + timedelta(days=int(x)) for d, x in zip(service_date, rng.integers(0, 10, size=N))]

member_age = rng.integers(18, 86, size=N)
member_gender = rng.choice(["F","M","U"], size=N, p=[0.49, 0.48, 0.03])
state = rng.choice(states, size=N)
plan_type = rng.choice(plan_types, size=N, p=[0.35, 0.45, 0.10, 0.10])

dx_grp = rng.choice(diagnosis_groups, size=N)
proc_grp = rng.choice(procedure_groups, size=N)
diagnosis_code = [pick_code("D") for _ in range(N)]
procedure_code = [pick_code("P") for _ in range(N)]

specialty = rng.choice(provider_specialties, size=N)
pos = rng.choice(place_of_service, size=N, p=[0.45,0.15,0.20,0.15,0.05])

# Utilization signals
prior_claims_90d = rng.poisson(2.0, size=N)
ed_visits_180d = rng.poisson(0.8, size=N)
chronic_index = np.clip(rng.normal(0.5, 0.25, size=N), 0, 1)  # 0..1
member_tenure_months = rng.integers(1, 120, size=N)

# Claim amounts (lognormal feels realistic)
claim_amount = np.round(rng.lognormal(mean=7.8, sigma=0.6, size=N), 2)  # ~ 500 to 10k+ range
allowed_amount = np.round(claim_amount * rng.uniform(0.55, 0.95, size=N), 2)

# Flags
prior_auth_required = rng.choice([0,1], size=N, p=[0.55, 0.45])
inpatient_flag = (pos == "Inpatient").astype(int)
out_of_network = rng.choice([0,1], size=N, p=[0.85, 0.15])

# Text fields
severity = rng.choice(["low","medium","high"], size=N, p=[0.45,0.40,0.15])

# Suspicion score (latent)
suspicious_pattern = (
    (claim_amount > np.quantile(claim_amount, 0.85)).astype(int) +
    (out_of_network == 1).astype(int) +
    (prior_claims_90d > 5).astype(int) +
    (ed_visits_180d > 3).astype(int)
)

# Add noise
fraud_signal_score = np.clip(
    0.15*suspicious_pattern + 0.25*(claim_amount/claim_amount.max()) + rng.normal(0,0.08,size=N),
    0, 1
)

clinical_notes = [
    make_note(dx, pr, sev, suspicious_pattern[i] >= 2)
    for i, (dx, pr, sev) in enumerate(zip(dx_grp, proc_grp, severity))
]

call_center_notes = [
    "Member called about claim status; asked for expedited review." if rng.random() < 0.22 else ""
    for _ in range(N)
]

# Introduce missingness (medium hard)
def apply_missing(arr, p):
    arr = np.array(arr, dtype=object)
    mask = rng.random(len(arr)) < p
    arr[mask] = None
    return arr

provider_id = apply_missing(provider_id, 0.02)
call_center_notes = apply_missing(call_center_notes, 0.15)
clinical_notes = apply_missing(clinical_notes, 0.05)
allowed_amount = allowed_amount.astype(object)
allowed_amount = apply_missing(allowed_amount, 0.03)  # sometimes missing

# --- Labels ---
# Fraud: rare, driven by fraud_signal_score + some categories
base_fraud_prob = 0.01 + 0.10*fraud_signal_score
boost = (
    0.02*(out_of_network==1) +
    0.01*(specialty=="Radiology") +
    0.01*(pos=="ER")
)
fraud_prob = np.clip(base_fraud_prob + boost, 0, 0.35)
fraud_label = (rng.random(N) < fraud_prob).astype(int)

# Review risk (higher -> clinician route)
review_risk_score = np.clip(
    0.35*(prior_auth_required==1) +
    0.20*(chronic_index) +
    0.15*(prior_claims_90d/10) +
    0.25*(claim_amount/claim_amount.max()) +
    0.10*(out_of_network==1) +
    rng.normal(0,0.08,size=N),
    0, 1
)

# Auto-approve label: safe claims that are low-risk and non-fraud
auto_approve_prob = np.clip(
    0.70
    - 0.85*review_risk_score
    - 0.60*fraud_signal_score
    + 0.10*(plan_type=="PPO")
    + rng.normal(0,0.05,size=N),
    0, 1
)
auto_approve_label = ((rng.random(N) < auto_approve_prob) & (fraud_label == 0)).astype(int)

df = pd.DataFrame({
    "claim_id": claim_id,
    "member_id": member_id,
    "provider_id": provider_id,
    "service_date": pd.to_datetime(service_date),
    "received_date": pd.to_datetime(received_date),
    "member_age": member_age,
    "member_gender": member_gender,
    "state": state,
    "plan_type": plan_type,
    "provider_specialty": specialty,
    "place_of_service": pos,
    "inpatient_flag": inpatient_flag,
    "out_of_network": out_of_network,
    "prior_auth_required": prior_auth_required,
    "prior_claims_90d": prior_claims_90d,
    "ed_visits_180d": ed_visits_180d,
    "chronic_index": np.round(chronic_index, 3),
    "member_tenure_months": member_tenure_months,
    "claim_amount": claim_amount,
    "allowed_amount": allowed_amount,
    "fraud_signal_score": np.round(fraud_signal_score, 3),
    "review_risk_score": np.round(review_risk_score, 3),
    "clinical_notes": clinical_notes,
    "call_center_notes": call_center_notes,
    "fraud_label": fraud_label,
    "auto_approve_label": auto_approve_label,
})

# Save
os.makedirs("data", exist_ok=True)
df.to_csv("data/claims_synth.csv", index=False)

df.head(), df.shape

(    claim_id member_id    provider_id service_date received_date  member_age  \
 0  CLM100000  MBR18032  NPI3343136890   2025-09-10    2025-09-16          76   
 1  CLM100001  MBR79655  NPI3902413968   2025-07-14    2025-07-22          23   
 2  CLM100002  MBR68910  NPI3182345677   2025-12-31    2026-01-06          68   
 3  CLM100003  MBR49498  NPI5318770604   2025-10-09    2025-10-14          52   
 4  CLM100004  MBR48970  NPI7149325218   2025-01-07    2025-01-11          69   
 
   member_gender state plan_type provider_specialty  ... chronic_index  \
 0             F    MA       HMO              Ortho  ...         0.507   
 1             F    IL       HMO          Radiology  ...         0.324   
 2             M    TX       PPO          Radiology  ...         0.454   
 3             F    CA       POS          Radiology  ...         0.674   
 4             F    MA       PPO         Cardiology  ...         0.758   
 
    member_tenure_months  claim_amount  allowed_amount  fraud_sign